# Robust / Uncertain-input Classification

39番Notebookの標準binary GP contractを前提に、label noiseとcandidate-time input uncertaintyを比較します。


In [ ]:
import torch

from robotorchan.models.classification import (
    ContinuousUncertainInputBinarySingleTaskGPClassifier,
    LabelNoiseBinarySingleTaskGPClassifier,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Label noise と input uncertainty

Label noiseは観測過程、input uncertaintyはcandidate入力の不確かさです。

In [ ]:
label_noise_model = LabelNoiseBinarySingleTaskGPClassifier(train_X, train_Y)
uncertain_model = ContinuousUncertainInputBinarySingleTaskGPClassifier(train_X, train_Y)
print(label_noise_model.classification_metadata)
print(uncertain_model.classification_metadata)


## 3. Candidate-time input uncertainty

In [ ]:
input_std = torch.full_like(grid, 0.03)
with torch.no_grad():
    probability = uncertain_model.predict_proba_with_input_uncertainty(
        grid, input_std=input_std, num_samples=32
    )
print(probability.shape)


## 4. 使い分け

- 誤ラベルはrobust likelihood側で扱います。
- candidate Xの揺らぎはuncertain-input側で扱います。
- Student-t regression likelihoodをbinary labelへ機械的に移植しません。
- categorical input uncertaintyには専用のexact marginalization APIがあります。